# Автономное управление посадкой (стенд)

Проект вынесен в пакет `ismpu/` (см. `implementation_plan.md`). Этот ноутбук — тонкая
обёртка: подключение к стенду Заказчика (ПИВ/ICS, UDP `0.0.0.0:3030`) и запуск
управляющего цикла. Стенд сам сообщает свой адрес в заголовке первого пакета.

**Управление идёт на всём интервале полёта.** Участок выбирается по первому кадру
телеметрии: если ВС в воздухе выше 400 футов — начинается заход по ILS с выравниванием
и касанием, дальше пробег; если ВС уже на полосе — сразу пробег (или подхват идущего).

Воздушный участок работает на статических коэффициентах (`ismpu/config/approach.py`),
перенесённых с живого стенда. Пресеты сценариев настраивают **пробег**: они подбираются
по фактическим отказам и погоде из `ICSInputs`. Доступные пресеты: `default`, `nws_fail`,
`left_reverse_fail`, `right_reverse_fail`, `right_wind`, `fwd_wind`, `wet_rwy`,
`puddly_rwy`, `icy_rwy` (`ismpu.envs.scenario.SCENARIO_PRESETS`).


In [ ]:
from ismpu.envs.scenario import SCENARIO_PRESETS, select_for_telemetry
from ismpu.runtime.loop import run
from ismpu.control.system import ControllingSystem
from ismpu.envs.ics_sim import ICSSim

In [ ]:
sim = ICSSim(listen_ip="0.0.0.0", listen_port=3030)
controller = ControllingSystem(sim)

# Пресет пробега подбирается по фактическим условиям стенда (погода + отказы из ICSInputs).
# Для отладки конкретного режима: scenario = SCENARIO_PRESETS["nws_fail"]
scenario = select_for_telemetry(sim.read_telemetry())
print("Сценарий пробега:", scenario.scenario_id)

scenario.apply_control(controller)


In [ ]:
# run() сам определяет участок полёта, проводит рукопожатие под него
# (ModeAIReady + переход ControlMode: 0 -> 1 в воздухе, 0 -> 4 на земле)
# и ведёт цикл 20 Гц до скорости руления.
# Выход по прерыванию ядра (Ctrl-C / Interrupt): органы обнуляются, заявка каналов снимается.
run(controller, sim, scenario)
